# Assignment 7 · The Sakila Stream Fleet
**ADSP 31011 · Assignment 7.** Last week you built the Sakila lakehouse by hand. This week it runs as a **fleet of fifteen Databricks jobs**, and new data keeps arriving: the stores' new point-of-sale system sends **eight monthly drops** of rentals and payments (March to October 2006, about **127,000 rental rows and 120,000 payments**, eight times the original Sakila). With every drop, the head office also sends **master-data changes**: customers who moved, switched store or closed their account; films whose price changed; copies that moved to the other store.

A new layer, **Silver Enriched**, keeps the full history of every dimension as a **slowly changing dimension (SCD Type 2)**, and a Gold table tells each customer's **journey**: where they lived, which store they used, and how much they rented in each chapter.

**This fleet does not entirely work.** There are **10 breaks** planted in the job notebooks: code errors, data errors, a streaming mistake, duplicates, silent logic bugs, and two in the SCD jobs. Some show up on the first drop; some only when a later drop arrives or when a job runs a second time. **Your job: take the fleet from red to green, keep it green until all eight drops are in, and answer the journey questions after drops 1 and 2.**

### What this notebook does
1. Creates the schemas `a7_bronze`, `a7_silver`, `a7_silver_enriched`, `a7_gold` and a landing volume
2. Lands the Sakila tables (packed in the notebook) and loads **Bronze**
3. Generates the eight monthly drops (rentals, payments and master-data changes) into a **staging** folder (same numbers for everyone)
4. Writes **fifteen pipeline notebooks** to your folder `a7_spark_fleet` and creates **fifteen jobs** (`A7 a1_silver_customers` ...)
5. Gives you a console: `land_next_drop()`, `run_fleet()`, `status()`, `why("job")`, `rerun("job")`, `reset_stream("rentals")`, `report_for_claude()`

### The loop you will repeat
**land a drop → run the fleet → read every red job → fix → rerun → green → land the next drop**

Run the cells one at a time. Steps 1 to 6 take about 5 minutes.

**DO NOT UPGRADE YOUR ACCOUNT TO A BILLING ACCOUNT.**

## Step 1 · Who you are and where things go

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs, workspace
import base64, io, zipfile, time
import pandas as pd

w = WorkspaceClient()
ME = w.current_user.me().user_name
CAT = "workspace"
FOLDER = f"/Users/{ME}/a7_spark_fleet"
VOL = "/Volumes/workspace/a7_bronze/landing"
PREFIX = "A7 "
print("You are:", ME)
print("Pipelines go in:", FOLDER)

You are: <redacted>
Pipelines go in: /Users/<redacted>/a7_spark_fleet


## Step 2 · Schemas and the landing volume
Also clears any earlier A7 run, so you always start from the beginning.

In [0]:
for s in ["a7_bronze", "a7_silver", "a7_silver_enriched", "a7_gold"]:
    spark.sql(f"DROP SCHEMA IF EXISTS {CAT}.{s} CASCADE") if s != "a7_bronze" else None
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAT}.{s}")
for t in ["rental_stream", "payment_stream", "control_totals"]:
    spark.sql(f"DROP TABLE IF EXISTS {CAT}.a7_bronze.{t}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CAT}.a7_bronze.landing")
for d in ["incoming", "_checkpoints", "staging"]:
    try:
        dbutils.fs.rm(f"{VOL}/{d}", True)
    except Exception:
        pass
for k in ["rentals", "payments", "customers", "films", "inventory"]:
    dbutils.fs.mkdirs(f"{VOL}/incoming/{k}")
display(spark.sql(f"SHOW SCHEMAS IN {CAT} LIKE 'a7_*'"))

databaseName
a7_bronze
a7_gold
a7_silver
a7_silver_enriched


## Step 3 · Land Sakila
The eleven Sakila tables are packed in this cell: nothing to upload.

In [0]:
# The eleven Sakila tables were packed here as one base64 zip (about 790 KB of text).
# Removed from this public copy: datasets are not stored in this repo. The original cell decoded
# the zip and uploaded each CSV to the landing volume:
B64 = "<base64 zip of the 11 Sakila CSVs, removed>"
with zipfile.ZipFile(io.BytesIO(base64.b64decode(B64))) as z:
    for name in z.namelist():
        w.files.upload(f"{VOL}/base/{name}", io.BytesIO(z.read(name)), overwrite=True)
        print("landed", name)

landed rental.csv
landed payment.csv
landed customer.csv
landed address.csv
landed city.csv
landed country.csv
landed inventory.csv
landed film.csv
landed film_category.csv
landed category.csv
landed store.csv


## Step 4 · Load Bronze (the 2005-06 history)

In [0]:
from pyspark.sql import functions as F
CAT = "workspace"
B = f"{CAT}.a7_bronze"
VOL = "/Volumes/workspace/a7_bronze/landing"
for t in ["rental", "payment", "customer", "address", "city", "country", "inventory", "film", "film_category", "category", "store"]:
    (spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv(f"{VOL}/base/{t}.csv")
     .withColumn("_source_file", F.col("_metadata.file_name")).withColumn("_ingested_at", F.current_timestamp())
     .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{B}.{t}_raw"))
    print(f"a7_bronze.{t}_raw".ljust(30), f"{spark.table(f'{B}.{t}_raw').count():>7,}", "rows")

a7_bronze.rental_raw            16,044 rows
a7_bronze.payment_raw           16,049 rows
a7_bronze.customer_raw             599 rows
a7_bronze.address_raw              603 rows
a7_bronze.city_raw                 600 rows
a7_bronze.country_raw              109 rows
a7_bronze.inventory_raw          4,581 rows
a7_bronze.film_raw               1,000 rows
a7_bronze.film_category_raw      1,000 rows
a7_bronze.category_raw              16 rows
a7_bronze.store_raw                  2 rows


## Step 4b · Generate the eight monthly drops
Writes March to October 2006 from the new point-of-sale system, and the head office's master-data changes, into `staging/`. Nothing is in `incoming/` yet: the drops land one at a time, when you call `land_next_drop()`. About 2 to 4 minutes.

In [0]:
from pyspark.sql import functions as F
CAT = "workspace"
VOL = "/Volumes/workspace/a7_bronze/landing"
N_DROPS, PER_DROP = 8, 15000
FMT_OLD, FMT_NEW = "yyyy-MM-dd HH:mm:ss", "MM/dd/yyyy HH:mm:ss"     # the POS system changes its date format from drop 3

def h(*parts):
    return F.abs(F.xxhash64(*parts))

def drop_rentals(d):
    """All rentals made in drop d's month (2006-03 + d-1), as the POS sends them."""
    start = F.unix_timestamp(F.lit(f"2006-{2 + d:02d}-01 00:00:00"))          # seconds since 1970
    return (spark.range(PER_DROP)
        .withColumn("rental_id", F.col("id") + 16050 + (d - 1) * PER_DROP)
        .withColumn("rental_ts", F.timestamp_seconds(start + F.pmod(h(F.col("rental_id"), F.lit("day")), F.lit(28)) * 86400
                                                     + F.pmod(h(F.col("rental_id"), F.lit("hr")), F.lit(24)) * 3600
                                                     + F.pmod(h(F.col("rental_id"), F.lit("mi")), F.lit(60)) * 60))
        .withColumn("open", F.pmod(h(F.col("rental_id"), F.lit("open")), F.lit(100)) < 6)          # still out when the drop is sent
        .withColumn("days", (1 + F.pmod(h(F.col("rental_id"), F.lit("ret")), F.lit(9))).cast("int"))
        .withColumn("late_days", (10 + F.pmod(h(F.col("rental_id"), F.lit("late")), F.lit(11))).cast("int"))
        .withColumn("inventory_id", (F.pmod(h(F.col("rental_id"), F.lit("inv")), F.lit(4581)) + 1))
        .withColumn("customer_id", (F.pmod(h(F.col("rental_id"), F.lit("cust")), F.lit(599)) + 1))
        .withColumn("staff_id", (F.pmod(h(F.col("rental_id"), F.lit("staff")), F.lit(2)) + 1)))

def plus_days(ts, days):
    return F.timestamp_seconds(F.unix_timestamp(ts) + days * 86400)

def fmt(col, d):
    return F.date_format(col, FMT_NEW if d >= 3 else FMT_OLD)

for d in range(1, N_DROPS + 1):
    cur = drop_rentals(d)
    rentals = cur.select("rental_id", fmt("rental_ts", d).alias("rental_date"), "inventory_id", "customer_id",
                         F.when(~F.col("open"), fmt(plus_days(F.col("rental_ts"), F.col("days")), d)).alias("return_date"),
                         "staff_id", fmt("rental_ts", d).alias("last_update"))
    if d > 1:     # last month's open rentals come back late, and the POS re-sends them with a return date
        prev = drop_rentals(d - 1).filter("open")
        rentals = rentals.unionByName(prev.select("rental_id", fmt("rental_ts", d).alias("rental_date"), "inventory_id", "customer_id",
                         fmt(plus_days(F.col("rental_ts"), F.col("late_days")), d).alias("return_date"),
                         "staff_id", fmt(plus_days(F.col("rental_ts"), F.col("late_days")), d).alias("last_update")))
    rentals = rentals.unionByName(rentals.filter(F.pmod(h(F.col("rental_id"), F.lit("dup")), F.lit(250)) == 0))   # the POS retries: some rows twice
    rentals.coalesce(2).write.mode("overwrite").option("header", True).csv(f"{VOL}/staging/drop_{d:02d}/rentals")

    pays = cur.select((F.col("rental_id")).alias("payment_id"), "customer_id", "staff_id", "rental_id",
                      F.format_number(F.element_at(F.array(F.lit(0.99), F.lit(2.99), F.lit(4.99)),
                                      (F.pmod(h(F.col("rental_id"), F.lit("rate")), F.lit(3)) + 1).cast("int"))
                                      + F.pmod(h(F.col("rental_id"), F.lit("fee")), F.lit(4)), 2).alias("amount"),
                      F.date_format("rental_ts", FMT_OLD).alias("payment_date"), F.date_format("rental_ts", FMT_OLD).alias("last_update"))
    pays.coalesce(2).write.mode("overwrite").option("header", True).csv(f"{VOL}/staging/drop_{d:02d}/payments")

    # ---- master-data changes from the head office, sent with the same drop (the slowly changing dimensions)
    stamp = f"2006-{2 + d:02d}-15 09:00:00"
    def bucket(col, salt):
        return F.pmod(h(F.col(col), F.lit(salt)), F.lit(100))
    def moved(addr, k):        # another existing customer address (Sakila has no address 257 or 518)
        m = F.pmod(addr.cast("int") + 97 * k, F.lit(599)) + 5
        return F.when(m.isin(257, 518), m + 1).otherwise(m).cast("string")
    cust = spark.table(f"{CAT}.a7_bronze.customer_raw").drop("_source_file", "_ingested_at")
    b = bucket("customer_id", "bucket")
    kind = F.pmod(h(F.col("customer_id"), F.lit("kind")), F.lit(4))     # 0 moved, 1 changed store, 2 closed account, 3 new email only
    once = (cust.filter((b >= 5 * (d - 1)) & (b < 5 * d))
                .withColumn("address_id", F.when(kind == 0, moved(F.col("address_id"), d)).otherwise(F.col("address_id")))
                .withColumn("store_id", F.when(kind == 1, (3 - F.col("store_id").cast("int")).cast("string")).otherwise(F.col("store_id")))
                .withColumn("active", F.when(kind == 2, F.lit("0")).otherwise(F.col("active")))
                .withColumn("email", F.when(kind == 3, F.concat(F.lower(F.concat_ws(".", "first_name", "last_name")), F.lit("@newmail.com"))).otherwise(F.col("email"))))
    custs = once
    if d in (1, 2):            # a few customers change twice: they move in drop 1, then also switch store in drop 2
        twice = (cust.filter((b >= 90) & (b < 92)).withColumn("address_id", moved(F.col("address_id"), 1)))
        if d == 2:
            twice = twice.withColumn("store_id", (3 - F.col("store_id").cast("int")).cast("string"))
        custs = custs.unionByName(twice)
    (custs.withColumn("last_update", F.lit(stamp))
          .coalesce(1).write.mode("overwrite").option("header", True).option("escape", '"').csv(f"{VOL}/staging/drop_{d:02d}/customers"))

    film = spark.table(f"{CAT}.a7_bronze.film_raw").drop("_source_file", "_ingested_at")
    fb = bucket("film_id", "fbucket")
    rate = F.col("rental_rate").cast("decimal(4,2)")
    films = film.filter((fb >= 3 * (d - 1)) & (fb < 3 * d)).withColumn("rental_rate", F.when(rate >= 4.99, rate - 1).otherwise(rate + 1).cast("string"))
    if d in (1, 2):            # a few films get a price rise in drop 1 and another in drop 2
        films = films.unionByName(film.filter((fb >= 95) & (fb < 97)).withColumn("rental_rate", (rate + d).cast("string")))
    (films.withColumn("last_update", F.lit(stamp))
          .coalesce(1).write.mode("overwrite").option("header", True).option("escape", '"').csv(f"{VOL}/staging/drop_{d:02d}/films"))

    inv = spark.table(f"{CAT}.a7_bronze.inventory_raw").drop("_source_file", "_ingested_at")
    ib = bucket("inventory_id", "ibucket")
    (inv.filter((ib >= 2 * (d - 1)) & (ib < 2 * d))
        .withColumn("store_id", (3 - F.col("store_id").cast("int")).cast("string")).withColumn("last_update", F.lit(stamp))
        .coalesce(1).write.mode("overwrite").option("header", True).option("escape", '"').csv(f"{VOL}/staging/drop_{d:02d}/inventory"))
    print(f"drop {d}: staged rentals, payments and master-data changes (customers, films, inventory) for 2006-{2 + d:02d}")

drop 1: staged rentals, payments and master-data changes (customers, films, inventory) for 2006-03
drop 2: staged rentals, payments and master-data changes (customers, films, inventory) for 2006-04
drop 3: staged rentals, payments and master-data changes (customers, films, inventory) for 2006-05
drop 4: staged rentals, payments and master-data changes (customers, films, inventory) for 2006-06
drop 5: staged rentals, payments and master-data changes (customers, films, inventory) for 2006-07
drop 6: staged rentals, payments and master-data changes (customers, films, inventory) for 2006-08
drop 7: staged rentals, payments and master-data changes (customers, films, inventory) for 2006-09
drop 8: staged rentals, payments and master-data changes (customers, films, inventory) for 2006-10


## Step 5 · Write the fifteen pipeline notebooks
Open the folder **a7_spark_fleet** afterwards: these are the notebooks you will fix.

In [0]:
PIPELINES = {
 "a1_silver_customers": {
  "wave": 1,
  "title": "Silver: customers with their city and country"
 },
 "a2_silver_films": {
  "wave": 1,
  "title": "Silver: films with their category"
 },
 "a3_silver_inventory": {
  "wave": 1,
  "title": "Silver: inventory"
 },
 "a4_stream_rentals": {
  "wave": 2,
  "title": "Stream: rentals from the point-of-sale drops"
 },
 "a5_stream_payments": {
  "wave": 2,
  "title": "Stream: payments from the point-of-sale drops"
 },
 "a6_silver_rentals": {
  "wave": 3,
  "title": "Silver: rentals (history + every drop, MERGE)"
 },
 "a7_silver_payments": {
  "wave": 3,
  "title": "Silver: payments (history + every drop)"
 },
 "e1_enriched_customers_scd2": {
  "wave": 3,
  "title": "Silver Enriched: customer history (SCD Type 2)"
 },
 "e2_enriched_films_scd2": {
  "wave": 3,
  "title": "Silver Enriched: film history (SCD Type 2)"
 },
 "e3_enriched_inventory_scd2": {
  "wave": 3,
  "title": "Silver Enriched: inventory copy history (SCD Type 2)"
 },
 "a8_gold_revenue_by_month_store": {
  "wave": 4,
  "title": "Gold: revenue by month and store"
 },
 "a9_gold_late_returns": {
  "wave": 4,
  "title": "Gold: late returns"
 },
 "a10_gold_top_films": {
  "wave": 4,
  "title": "Gold: top 3 films per category per month"
 },
 "a12_gold_customer_journeys": {
  "wave": 4,
  "title": "Gold: customer journeys (rentals in each version)"
 },
 "a11_quality_gate": {
  "wave": 5,
  "title": "Quality gate"
 }
}

# SOURCES = { ... }  # the fifteen course-provided job notebooks (with the planted bugs) were packed here.
# Removed from this public copy; the fixed versions are in pipelines/ with each change marked # FIX.

w.workspace.mkdirs(FOLDER)
for name, src in SOURCES.items():
    w.workspace.import_(path=f"{FOLDER}/{name}", content=base64.b64encode(src.encode()).decode(),
                        format=workspace.ImportFormat.SOURCE, language=workspace.Language.PYTHON, overwrite=True)
    print("wrote", f"{FOLDER}/{name}")

wrote /Users/<redacted>/a7_spark_fleet/a1_silver_customers
wrote /Users/<redacted>/a7_spark_fleet/a2_silver_films
wrote /Users/<redacted>/a7_spark_fleet/a3_silver_inventory
wrote /Users/<redacted>/a7_spark_fleet/a4_stream_rentals
wrote /Users/<redacted>/a7_spark_fleet/a5_stream_payments
wrote /Users/<redacted>/a7_spark_fleet/a6_silver_rentals
wrote /Users/<redacted>/a7_spark_fleet/a7_silver_payments
wrote /Users/<redacted>/a7_spark_fleet/e1_enriched_customers_scd2
wrote /Users/<redacted>/a7_spark_fleet/e2_enriched_films_scd2
wrote /Users/<redacted>/a7_spark_fleet/e3_enriched_inventory_scd2
wrote /Users/<redacted>/a7_spark_fleet/a8_gold_revenue_by_month_store
wrote /Users/<redacted>/a7_spark_fleet/a9_gold_late_returns
wrote /Users/<redacted>/a7_spark_fleet/a10_gold_top_films
wrote /Users/<redacted>/a7_spark_fleet/a12_gold_customer_journeys
wrote /Users/<redacted>/a7_spark_fleet/a11_quality_gate


## Step 6 · Create the jobs
One job per pipeline. Re-running this cell replaces them. Automatic retries are **off** (`disable_auto_optimization`, `max_retries=0`): serverless would otherwise retry failed tasks and hide the failures.

In [0]:
JOB_IDS = {}
for name, meta in PIPELINES.items():
    for old in w.jobs.list(name=PREFIX + name):
        w.jobs.delete(job_id=old.job_id)
    nbt = jobs.NotebookTask(notebook_path=f"{FOLDER}/{name}", source=jobs.Source.WORKSPACE)
    try:      # no automatic retries: we want to SEE the failures
        task = jobs.Task(task_key="run", notebook_task=nbt, max_retries=0, disable_auto_optimization=True)
    except TypeError:
        task = jobs.Task(task_key="run", notebook_task=nbt, max_retries=0)
    j = w.jobs.create(
        name=PREFIX + name,
        tags={"course": "adsp31011", "lab": "a7", "wave": str(meta["wave"])},
        max_concurrent_runs=1,
        tasks=[task])
    JOB_IDS[name] = j.job_id
    print(f"created {PREFIX + name:36s} job_id {j.job_id}")

created A7 a1_silver_customers               job_id 710265812433585
created A7 a2_silver_films                   job_id 951869626049333
created A7 a3_silver_inventory               job_id 810055851897787
created A7 a4_stream_rentals                 job_id 461306335646986
created A7 a5_stream_payments                job_id 552032721474395
created A7 a6_silver_rentals                 job_id 817981052019380
created A7 a7_silver_payments                job_id 347269965683305
created A7 e1_enriched_customers_scd2        job_id 73657583540158
created A7 e2_enriched_films_scd2            job_id 222812059686796
created A7 e3_enriched_inventory_scd2        job_id 264842977551938
created A7 a8_gold_revenue_by_month_store    job_id 179315144696192
created A7 a9_gold_late_returns              job_id 238008891824014
created A7 a10_gold_top_films                job_id 860148089812808
created A7 a12_gold_customer_journeys        job_id 226904438136568
created A7 a11_quality_gate                  job_

## Step 7 · The console
| Call | What it does |
|---|---|
| `land_next_drop()` | releases the next monthly drop into `incoming/` and updates the control totals |
| `run_fleet()` | runs every job in order: Silver dimensions (1), streams (2), Silver facts + Silver Enriched history (3), Gold (4), quality gate (5) |
| `status()` | one row per job: green or red |
| `why("a6_silver_rentals")` | the error from that job's last run |
| `rerun("a6_silver_rentals")` | runs one job again, after you fix its notebook |
| `reset_stream("rentals")` | empties a stream's Bronze table and checkpoint, so it replays every landed file exactly once |
| `report_for_claude()` | every red job with its error and its code, ready to paste into Claude |

In [0]:
def _job_ids():
    ids = {}
    for name in PIPELINES:
        found = list(w.jobs.list(name=PREFIX + name))
        if found:
            ids[name] = found[0].job_id
    return ids

def _wait(run_ids, timeout_min=20):
    t0 = time.time()
    pending = set(run_ids)
    while pending and time.time() - t0 < timeout_min * 60:
        for rid in list(pending):
            st = w.jobs.get_run(run_id=rid).state
            if st.life_cycle_state.value in ("TERMINATED", "SKIPPED", "INTERNAL_ERROR"):
                pending.discard(rid)
        if pending:
            time.sleep(10)

def _start(name):
    wt = w.jobs.run_now(job_id=_job_ids()[name])
    try:
        return wt.response.run_id
    except AttributeError:
        return wt.run_id

def rerun(name):
    print("running", name, "...")
    _wait([_start(name)])
    return status()

def run_fleet():
    for wave in sorted(set(m["wave"] for m in PIPELINES.values())):
        names = [n for n, m in PIPELINES.items() if m["wave"] == wave]
        print(f"wave {wave}: {', '.join(names)}")
        _wait([_start(n) for n in names])
    return status()

def _last_run(name):
    runs = list(w.jobs.list_runs(job_id=_job_ids()[name], limit=1))
    return runs[0] if runs else None

def status():
    rows = []
    for name, meta in PIPELINES.items():
        r = _last_run(name)
        if r is None:
            rows.append((meta["wave"], name, "not run", None, "")); continue
        st = r.state
        res = st.result_state.value if st.result_state else st.life_cycle_state.value
        light = "GREEN" if res == "SUCCESS" else ("RED" if res in ("FAILED", "TIMEDOUT", "INTERNAL_ERROR") else res)
        secs = round((r.run_duration or r.execution_duration or 0) / 1000)
        rows.append((meta["wave"], name, light, secs, r.run_page_url))
    df = pd.DataFrame(rows, columns=["wave", "job", "status", "seconds", "run page"])
    print(f"{(df.status == 'GREEN').sum()} of {len(df)} green")
    return df

def why(name, chars=1500):
    r = _last_run(name)
    if r is None:
        return print(name, "has not run yet")
    run = w.jobs.get_run(run_id=r.run_id)
    out = w.jobs.get_run_output(run_id=run.tasks[0].run_id)
    msg = out.error or run.state.state_message or "(no error: this job succeeded)"
    print(f"{name}: {msg[:chars]}")
    return msg

def report_for_claude():
    lines = ["Databricks job fleet report (ADSP 31011 Assignment 7). Red jobs, with the error and the notebook code:", ""]
    for name, meta in PIPELINES.items():
        r = _last_run(name)
        if r is None or (r.state.result_state and r.state.result_state.value == "SUCCESS"):
            continue
        run = w.jobs.get_run(run_id=r.run_id)
        out = w.jobs.get_run_output(run_id=run.tasks[0].run_id)
        code = base64.b64decode(w.workspace.export(f"{FOLDER}/{name}", format=workspace.ExportFormat.SOURCE).content).decode()
        lines += [f"=== {name} (wave {meta['wave']}) ===", "ERROR: " + (out.error or run.state.state_message or "")[:800],
                  "CODE:", code, ""]
    text = "\\n".join(lines)
    print(text)
    return None

CONTROL = {"1": {"drop": 1, "customers": 599, "films": 1000, "inventory": 4581, "rentals_landed": 15061, "payments_landed": 15000, "rentals": 31044, "open_rentals": 1074, "payments": 31049, "revenue": "134997.51", "gold_revenue": "134987.56", "late_returns": 13695, "top_films_rows": 288, "customer_versions": 632, "film_versions": 1042, "inventory_versions": 4679}, "2": {"drop": 2, "customers": 599, "films": 1000, "inventory": 4581, "rentals_landed": 31008, "payments_landed": 30000, "rentals": 46044, "open_rentals": 1075, "payments": 46049, "revenue": "202270.51", "gold_revenue": "202260.56", "late_returns": 20898, "top_films_rows": 336, "customer_versions": 667, "film_versions": 1107, "inventory_versions": 4771}, "3": {"drop": 3, "customers": 599, "films": 1000, "inventory": 4581, "rentals_landed": 46976, "payments_landed": 45000, "rentals": 61044, "open_rentals": 1113, "payments": 61049, "revenue": "269756.51", "gold_revenue": "269746.56", "late_returns": 28149, "top_films_rows": 384, "customer_versions": 688, "film_versions": 1142, "inventory_versions": 4865}, "4": {"drop": 4, "customers": 599, "films": 1000, "inventory": 4581, "rentals_landed": 62974, "payments_landed": 60000, "rentals": 76044, "open_rentals": 1072, "payments": 76049, "revenue": "336794.51", "gold_revenue": "336784.56", "late_returns": 35489, "top_films_rows": 432, "customer_versions": 709, "film_versions": 1170, "inventory_versions": 4949}, "5": {"drop": 5, "customers": 599, "films": 1000, "inventory": 4581, "rentals_landed": 78930, "payments_landed": 75000, "rentals": 91044, "open_rentals": 1085, "payments": 91049, "revenue": "404238.51", "gold_revenue": "404228.56", "late_returns": 42806, "top_films_rows": 480, "customer_versions": 732, "film_versions": 1203, "inventory_versions": 5037}, "6": {"drop": 6, "customers": 599, "films": 1000, "inventory": 4581, "rentals_landed": 94896, "payments_landed": 90000, "rentals": 106044, "open_rentals": 1132, "payments": 106049, "revenue": "471740.51", "gold_revenue": "471730.56", "late_returns": 50072, "top_films_rows": 528, "customer_versions": 750, "film_versions": 1234, "inventory_versions": 5137}, "7": {"drop": 7, "customers": 599, "films": 1000, "inventory": 4581, "rentals_landed": 110912, "payments_landed": 105000, "rentals": 121044, "open_rentals": 1036, "payments": 121049, "revenue": "538677.51", "gold_revenue": "538667.56", "late_returns": 57445, "top_films_rows": 576, "customer_versions": 773, "film_versions": 1255, "inventory_versions": 5232}, "8": {"drop": 8, "customers": 599, "films": 1000, "inventory": 4581, "rentals_landed": 126830, "payments_landed": 120000, "rentals": 136044, "open_rentals": 1046, "payments": 136049, "revenue": "606116.51", "gold_revenue": "606106.56", "late_returns": 64707, "top_films_rows": 624, "customer_versions": 796, "film_versions": 1289, "inventory_versions": 5303}}

def _current_drop():
    try:
        return int({r.metric: r.value for r in spark.table(f"{CAT}.a7_bronze.control_totals").collect()}["drop"])
    except Exception:
        return 0

def land_next_drop():
    d = _current_drop() + 1
    if d > 8:
        return print("All eight drops have landed.")
    for kind in ["rentals", "payments", "customers", "films", "inventory"]:
        for f in dbutils.fs.ls(f"{VOL}/staging/drop_{d:02d}/{kind}"):
            if f.name.endswith(".csv"):
                dbutils.fs.cp(f.path, f"{VOL}/incoming/{kind}/drop_{d:02d}_{f.name}")
    rows = [(k, str(v)) for k, v in CONTROL[str(d)].items()]
    spark.createDataFrame(rows, "metric string, value string").write.mode("overwrite").saveAsTable(f"{CAT}.a7_bronze.control_totals")
    print(f"Drop {d} landed: 2006-{2 + d:02d}. Control totals now:")
    display(spark.table(f"{CAT}.a7_bronze.control_totals"))

def reset_stream(kind):
    table = {"rentals": "rental_stream", "payments": "payment_stream"}[kind]
    spark.sql(f"DROP TABLE IF EXISTS {CAT}.a7_bronze.{table}")
    for f in dbutils.fs.ls(f"{VOL}/_checkpoints") if _exists(f"{VOL}/_checkpoints") else []:
        if f.name.startswith(kind):
            dbutils.fs.rm(f.path, True)
    print(f"reset: {table} dropped and its checkpoints cleared. Rerun the stream job to replay every landed file once.")

def _exists(path):
    try:
        dbutils.fs.ls(path); return True
    except Exception:
        return False

## Step 8 · Drop 1 arrives: land it and run the fleet
**EVIDENCE E1:** screenshot the status table from this first run (the red fleet) with your user icon showing.

In [0]:
land_next_drop()

Drop 1 landed: 2006-03. Control totals now:


metric,value
drop,1
customers,599
films,1000
inventory,4581
rentals_landed,15061
payments_landed,15000
rentals,31044
open_rentals,1074
payments,31049
revenue,134997.51


In [0]:
run_fleet()

wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
15 of 15 green


,wave,job,status,seconds,run page
0,1,a1_silver_customers,GREEN,45,run page (redacted)
1,1,a2_silver_films,GREEN,43,run page (redacted)
2,1,a3_silver_inventory,GREEN,40,run page (redacted)
3,2,a4_stream_rentals,GREEN,38,run page (redacted)
4,2,a5_stream_payments,GREEN,37,run page (redacted)
5,3,a6_silver_rentals,GREEN,37,run page (redacted)
6,3,a7_silver_payments,GREEN,22,run page (redacted)
7,3,e1_enriched_customers_scd2,GREEN,33,run page (redacted)
8,3,e2_enriched_films_scd2,GREEN,33,run page (redacted)
9,3,e3_enriched_inventory_scd2,GREEN,30,run page (redacted)


In [0]:
why("a4_stream_rentals")

a4_stream_rentals: AssertionError: Bronze has 30,122 streamed rental rows, but 15,061 have landed


'AssertionError: Bronze has 30,122 streamed rental rows, but 15,061 have landed'

In [0]:
reset_stream("rentals")

reset: rental_stream dropped and its checkpoints cleared. Rerun the stream job to replay every landed file once.


## Step 9 · Ask Claude, before you fix anything
Run the next cell, copy everything it prints, and paste it into Claude with:

> *Here is the report from my Databricks job fleet. For each red job, tell me what kind of failure it is (transient, cascade from another job, code error, data error, schema or format change, duplicates or re-runs, silent logic bug), what the evidence is, the exact line, and the order to fix them in.*

Claude can read all of it, and in VS Code it can read your whole `a7_spark_fleet` folder and the job runs through the Databricks CLI (see **Practice 3** in the Week 7 module). Let it explain as much as you like; **you make each fix** and confirm the CHECK goes green. Keep the conversation going while you work, and save it for your submission. You can also use the Databricks **Assistant** (*Diagnose error*) on a red job, and a **Genie** space over `a7_silver_enriched` and `a7_gold` for the journey questions.

In [0]:
report_for_claude()

Databricks job fleet report (ADSP 31011 Assignment 7). Red jobs, with the error and the notebook code:

=== a1_silver_customers (wave 1) ===
ERROR: [UNRESOLVED_USING_COLUMN_FOR_JOIN] USING column `city_id` cannot be resolved on the right side of the join. The right-side columns: [`country`, `country_id`]. SQLSTATE: 42703
CODE: (notebook source omitted in this public copy; see pipelines/ for the fixed versions)
=== a4_stream_rentals (wave 2) ===
ERROR: TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'
CODE: (notebook source omitted in this public copy; see pipelines/ for the fixed versions)
=== a5_stream_payments (wave 2) ===
ERROR: TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'
CODE: (notebook source omitted in this public copy; see pipelines/ for the fixed versions)
=== a6_silver_rentals (wave 3) ===
ERROR: AssertionError: 61 rental_ids appear more than once in Silver
CODE: (notebook source omitted in this public copy; see pipelines/ for the fixed 

## Step 10 · Troubleshoot to green
For each red job, working from wave 1 down:

1. `why("job")`, or open the job's run page, and read the error. Decide what kind of failure it is.
2. Open the notebook in **a7_spark_fleet**, find the cause, make the **smallest** change that fixes it.
3. `rerun("job")`, then rerun the jobs downstream of it. When in doubt, `run_fleet()`.
4. Write a row in your **fix log**: job · error class or failed check · the cause in one sentence · the line you changed.

Things to know:
* **Some jobs have two problems.** The second shows up after you fix the first, or only when a later drop arrives.
* **Some bugs raise no Spark error.** The CHECK cell at the bottom of each notebook compares the job's output with the control totals. A failed CHECK is a real bug.
* **Never** make a job green by deleting or weakening its CHECK. That is the one fix that is always wrong.
* The two stream jobs (a4, a5) do the same thing for different data. If one misbehaves, compare it with the other.
* If a stream job has written the same files twice, fix it, then run `reset_stream("rentals")` (or `"payments"`) once and rerun it.
* The three **Silver Enriched** jobs (e1, e2, e3) build history the same way. If one misbehaves, compare it with the others. Their CHECKs test three rules of a good SCD Type 2: one current version per key, **no overlapping versions**, and the right number of versions.

In [0]:
why("a1_silver_customers")

a1_silver_customers: (no error: this job succeeded)


'(no error: this job succeeded)'

In [0]:
status()

15 of 15 green


,wave,job,status,seconds,run page
0,1,a1_silver_customers,GREEN,45,run page (redacted)
1,1,a2_silver_films,GREEN,43,run page (redacted)
2,1,a3_silver_inventory,GREEN,40,run page (redacted)
3,2,a4_stream_rentals,GREEN,38,run page (redacted)
4,2,a5_stream_payments,GREEN,37,run page (redacted)
5,3,a6_silver_rentals,GREEN,37,run page (redacted)
6,3,a7_silver_payments,GREEN,22,run page (redacted)
7,3,e1_enriched_customers_scd2,GREEN,33,run page (redacted)
8,3,e2_enriched_films_scd2,GREEN,33,run page (redacted)
9,3,e3_enriched_inventory_scd2,GREEN,30,run page (redacted)


## Step 10b · Journey questions, after drop 1 (answer in your write-up)
When the fleet is **green after drop 1**, answer each question with **one short piece of Spark** (`filter`, `select`, `join`, `groupBy`, `orderBy`), and paste the code and the answer into your write-up. Tables: `a7_silver_enriched.customers_scd2`, `films_scd2`, `inventory_scd2`, `a7_gold.customer_journeys`, `a7_silver.rental`.

1. How many customers now have **more than one version**? How many of them changed in drop 1 (valid_from = 2006-03-15)?
2. How many customers **moved to a different city** in drop 1? Pick the one with the smallest customer_id and tell their journey in one sentence: who, from where, to where, from what date.
3. How many films had their **price changed** in drop 1, and how many of those got **cheaper**?
4. How many copies **moved store** in drop 1, and how many went to each store?
5. How many rentals are **still out** (no return date) after drop 1?
6. **Point out:** the head office also sent some changes that did **not** become a new version. Find one in the landed files (`incoming/customers`) and say why it is not a version.

In [0]:
from pyspark.sql import functions as F
cs  = spark.table(f"{CAT}.a7_silver_enriched.customers_scd2")
fs  = spark.table(f"{CAT}.a7_silver_enriched.films_scd2")
inv = spark.table(f"{CAT}.a7_silver_enriched.inventory_scd2")
jr  = spark.table(f"{CAT}.a7_gold.customer_journeys")

def journey(customer_id):
    display(jr.filter(F.col("customer_id") == customer_id).orderBy("version"))

# example: every version of customer 1, oldest first
display(cs.filter(F.col("customer_id") == 1).orderBy("version"))

address_id,customer_id,first_name,last_name,store_id,active,email,city,country,version,valid_from,valid_to,is_current
5,1,MARY,SMITH,1,1,MARY.SMITH@sakilacustomer.org,Sasebo,Japan,1,2005-01-01,null,true


In [0]:
# your drop 1 answers here
from pyspark.sql import functions as F
D1 = "2006-03-15"   # drop 1's change date

# helper: pair each version with the one before it
def with_prev(t, key):
    n, o = t.alias("n"), t.alias("o")
    return n.join(o, (F.col(f"n.{key}") == F.col(f"o.{key}")) & (F.col("n.version") == F.col("o.version") + 1))

# Q1 customers with more than one version / how many changed in drop 1
q1a = cs.groupBy("customer_id").count().filter("count > 1").count()
q1b = cs.filter(F.col("valid_from") == D1).select("customer_id").distinct().count()
print("Q1:", q1a, "customers with >1 version;", q1b, "changed in drop 1")

# Q2 moved to a different city in drop 1; the smallest customer_id's journey
moved = (with_prev(cs, "customer_id")
         .filter((F.col("n.valid_from") == D1) & (F.col("n.city") != F.col("o.city")))
         .select(F.col("n.customer_id").alias("customer_id"), "n.first_name", "n.last_name",
                 F.col("o.city").alias("from_city"), F.col("o.country").alias("from_country"),
                 F.col("n.city").alias("to_city"), F.col("n.country").alias("to_country"), "n.valid_from")
         .orderBy("customer_id"))
print("Q2:", moved.count(), "customers moved city"); display(moved.limit(1))

# Q3 films with a price change in drop 1 / how many got cheaper
price = (with_prev(fs, "film_id").filter((F.col("n.valid_from") == D1) & (F.col("n.rental_rate") != F.col("o.rental_rate"))))
print("Q3:", price.count(), "price changes;", price.filter(F.col("n.rental_rate") < F.col("o.rental_rate")).count(), "cheaper")

# Q4 copies that moved store in drop 1, by destination store
display(inv.filter(F.col("valid_from") == D1).groupBy("store_id").count().orderBy("store_id"))

# Q5 rentals still out after drop 1
print("Q5:", spark.table(f"{CAT}.a7_silver.rental").filter(F.col("return_date").isNull()).count(), "still out")

# Q6 a change that did NOT become a version: look at the landed customer change file
chg = spark.read.option("header", True).csv(f"{VOL}/incoming/customers/")
display(chg.filter(F.col("email").endswith("@newmail.com")).select("customer_id", "first_name", "last_name", "email", "store_id", "address_id", "active", "last_update"))

Q1: 33 customers with >1 version; 33 changed in drop 1
Q2: 16 customers moved city


customer_id,first_name,last_name,from_city,from_country,to_city,to_country,valid_from
49,JOYCE,EDWARDS,Jedda,Saudi Arabia,Laiwu,China,2006-03-15


Q3: 42 price changes; 3 cheaper


store_id,count
1,44
2,54


Q5: 1074 still out


customer_id,first_name,last_name,email,store_id,address_id,active,last_update
5,ELIZABETH,BROWN,elizabeth.brown@newmail.com,1,9,1,2006-03-15 09:00:00
210,ELLA,OLIVER,ella.oliver@newmail.com,2,214,1,2006-03-15 09:00:00
326,JOSE,ANDREW,jose.andrew@newmail.com,1,331,1,2006-03-15 09:00:00
373,LOUIS,LEONE,louis.leone@newmail.com,1,378,1,2006-03-15 09:00:00
586,KIRK,STCLAIR,kirk.stclair@newmail.com,1,592,1,2006-03-15 09:00:00


In [0]:
#Q6 check:
display(cs.filter("customer_id = 5"))

address_id,customer_id,first_name,last_name,store_id,active,email,city,country,version,valid_from,valid_to,is_current
9,5,ELIZABETH,BROWN,1,1,ELIZABETH.BROWN@sakilacustomer.org,Nantou,Taiwan,1,2005-01-01,null,true


In [0]:
#Q3 check:
display(price.groupBy(F.col("o.rental_rate").alias("old"), F.col("n.rental_rate").alias("new")).count())

old,new,count
2.99,3.99,18
4.99,5.99,8
0.99,1.99,13
4.99,3.99,3


## Step 11 · Land the rest of the drops
When the whole fleet is green, land the next drop and run again. **Run the fleet twice after each drop**: a correct pipeline gives the same green both times. Some breaks only appear at a particular drop: fix them, log them, keep going.

In [0]:
land_next_drop()
run_fleet()

Drop 2 landed: 2006-04. Control totals now:


metric,value
drop,2
customers,599
films,1000
inventory,4581
rentals_landed,31008
payments_landed,30000
rentals,46044
open_rentals,1075
payments,46049
revenue,202270.51


wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
15 of 15 green


,wave,job,status,seconds,run page
0,1,a1_silver_customers,GREEN,43,run page (redacted)
1,1,a2_silver_films,GREEN,41,run page (redacted)
2,1,a3_silver_inventory,GREEN,40,run page (redacted)
3,2,a4_stream_rentals,GREEN,22,run page (redacted)
4,2,a5_stream_payments,GREEN,20,run page (redacted)
5,3,a6_silver_rentals,GREEN,60,run page (redacted)
6,3,a7_silver_payments,GREEN,48,run page (redacted)
7,3,e1_enriched_customers_scd2,GREEN,55,run page (redacted)
8,3,e2_enriched_films_scd2,GREEN,51,run page (redacted)
9,3,e3_enriched_inventory_scd2,GREEN,50,run page (redacted)


In [0]:
run_fleet()      # the second run: still green?

wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
15 of 15 green


,wave,job,status,seconds,run page
0,1,a1_silver_customers,GREEN,44,run page (redacted)
1,1,a2_silver_films,GREEN,44,run page (redacted)
2,1,a3_silver_inventory,GREEN,41,run page (redacted)
3,2,a4_stream_rentals,GREEN,55,run page (redacted)
4,2,a5_stream_payments,GREEN,54,run page (redacted)
5,3,a6_silver_rentals,GREEN,58,run page (redacted)
6,3,a7_silver_payments,GREEN,45,run page (redacted)
7,3,e1_enriched_customers_scd2,GREEN,52,run page (redacted)
8,3,e2_enriched_films_scd2,GREEN,51,run page (redacted)
9,3,e3_enriched_inventory_scd2,GREEN,50,run page (redacted)


## Step 11b · Journey questions, after drop 2 (answer in your write-up)
Answer these **right after drop 2 is green**, before you land drop 3: the answers change as more drops land.

7. Which customers now have **three versions**? Tell the journey of customer **49**: each version, what changed, and how many rentals they made in each chapter (`journey(49)`).
8. **Point in time:** what was film **29**'s rental rate on **2006-03-10**, on **2006-03-31**, and on **2006-04-20**? (filter `valid_from <= day AND (valid_to IS NULL OR valid_to >= day)`). Each answer must be exactly **one** row.
9. How many customers have a version with `active = 0` (account closed) that still has **rentals** in it? Is that a data problem or a business story? One sentence.
10. **Point out:** `a8_gold_revenue_by_month_store` credits each payment to the store that owns the copy **today** (`a7_silver.inventory`). Copies move between stores. Which table would credit the store that owned the copy **on the day of the rental**? Describe the join in one sentence (you do not have to build it).

In [0]:
# your drop 2 answers here 
from pyspark.sql import functions as F

# Q7 customers with three versions, and customer 49's journey
three = cs.groupBy("customer_id").count().filter("count = 3").orderBy("customer_id")
print("Q7:", three.count(), "customers with 3 versions"); display(three)
journey(49)

# Q8 film 29's rental_rate on three days (each must be exactly one row)
from functools import reduce
days = ["2006-03-10", "2006-03-31", "2006-04-20"]
q8 = reduce(lambda a, b: a.unionByName(b), [
        fs.filter((F.col("film_id") == 29) & (F.col("valid_from") <= d) & (F.col("valid_to").isNull() | (F.col("valid_to") >= d)))
          .select(F.lit(d).alias("day"), "version", "valid_from", "valid_to", "rental_rate") for d in days])
display(q8.orderBy("day"))

# Q9 customers with a closed-account version (active = 0) that still has rentals in it
closed = jr.filter((F.col("active") == 0) & (F.col("rentals") > 0))
print("Q9:", closed.select("customer_id").distinct().count(), "customers")
display(closed.groupBy("version", "valid_from").agg(F.countDistinct("customer_id").alias("customers"), F.sum("rentals").alias("rentals")).orderBy("valid_from"))

Q7: 10 customers with 3 versions


customer_id,count
49,3
52,3
128,3
183,3
314,3
349,3
420,3
430,3
444,3
484,3


customer_id,version,valid_from,valid_to,is_current,store_id,city,country,active,rentals,paid,first_rental,last_rental
49,1,2005-01-01,2006-03-14,false,2,Jedda,Saudi Arabia,1,41,187.59,2005-05-25,2006-03-14
49,2,2006-03-15,2006-04-14,false,2,Laiwu,China,1,34,150.66,2006-03-15,2006-04-13
49,3,2006-04-15,null,true,1,Laiwu,China,1,9,44.91,2006-04-15,2006-04-28


day,version,valid_from,valid_to,rental_rate
2006-03-10,1,2005-01-01,2006-03-14,2.99
2006-03-31,2,2006-03-15,2006-04-14,3.99
2006-04-20,3,2006-04-15,null,4.99


Q9: 34 customers


version,valid_from,customers,rentals
1,2005-01-01,15,1122
2,2006-03-15,11,419
2,2006-04-15,9,125


Repeat the two cells above until all eight drops have landed. A shortcut once you trust your fixes:

In [0]:
for _ in range(8):
    if _current_drop() >= 8:
        break
    land_next_drop()
    df = run_fleet()
    if (df.status != "GREEN").any():
        print("Stopped: something went red at drop", _current_drop()); break
status()

Drop 3 landed: 2006-05. Control totals now:


metric,value
drop,3
customers,599
films,1000
inventory,4581
rentals_landed,46976
payments_landed,45000
rentals,61044
open_rentals,1113
payments,61049
revenue,269756.51


wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
9 of 15 green
Stopped: something went red at drop 3
9 of 15 green


,wave,job,status,seconds,run page
0,1,a1_silver_customers,GREEN,59,run page (redacted)
1,1,a2_silver_films,GREEN,57,run page (redacted)
2,1,a3_silver_inventory,GREEN,56,run page (redacted)
3,2,a4_stream_rentals,GREEN,26,run page (redacted)
4,2,a5_stream_payments,GREEN,25,run page (redacted)
5,3,a6_silver_rentals,RED,31,run page (redacted)
6,3,a7_silver_payments,GREEN,22,run page (redacted)
7,3,e1_enriched_customers_scd2,GREEN,36,run page (redacted)
8,3,e2_enriched_films_scd2,GREEN,34,run page (redacted)
9,3,e3_enriched_inventory_scd2,GREEN,32,run page (redacted)


In [0]:
why("a6_silver_rentals")

a6_silver_rentals: [CANNOT_PARSE_TIMESTAMP] Text '05/14/2006 22:33:00' could not be parsed at index 0. Use `try_to_timestamp` to tolerate invalid input string and return NULL instead. SQLSTATE: 22007


"[CANNOT_PARSE_TIMESTAMP] Text '05/14/2006 22:33:00' could not be parsed at index 0. Use `try_to_timestamp` to tolerate invalid input string and return NULL instead. SQLSTATE: 22007"

In [0]:
run_fleet()

wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
15 of 15 green


,wave,job,status,seconds,run page
0,1,a1_silver_customers,GREEN,44,run page (redacted)
1,1,a2_silver_films,GREEN,43,run page (redacted)
2,1,a3_silver_inventory,GREEN,42,run page (redacted)
3,2,a4_stream_rentals,GREEN,35,run page (redacted)
4,2,a5_stream_payments,GREEN,34,run page (redacted)
5,3,a6_silver_rentals,GREEN,56,run page (redacted)
6,3,a7_silver_payments,GREEN,45,run page (redacted)
7,3,e1_enriched_customers_scd2,GREEN,52,run page (redacted)
8,3,e2_enriched_films_scd2,GREEN,47,run page (redacted)
9,3,e3_enriched_inventory_scd2,GREEN,47,run page (redacted)


In [0]:
for _ in range(8):
    if _current_drop() >= 8:
        break
    land_next_drop()
    for attempt in (1, 2):                  # run the fleet twice per drop
        df = run_fleet()
        if (df.status != "GREEN").any():
            break
    if (df.status != "GREEN").any():
        print("Stopped: something went red at drop", _current_drop(), "run", attempt); break
status()

Drop 4 landed: 2006-06. Control totals now:


metric,value
drop,4
customers,599
films,1000
inventory,4581
rentals_landed,62974
payments_landed,60000
rentals,76044
open_rentals,1072
payments,76049
revenue,336794.51


wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
15 of 15 green
wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
15 of 15 green
Drop 5 landed: 2006-07. Control totals now:


metric,value
drop,5
customers,599
films,1000
inventory,4581
rentals_landed,78930
payments_landed,75000
rentals,91044
open_rentals,1085
payments,91049
revenue,404238.51


wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
15 of 15 green
wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
15 of 15 green
Drop 6 landed: 2006-08. Control totals now:


metric,value
drop,6
customers,599
films,1000
inventory,4581
rentals_landed,94896
payments_landed,90000
rentals,106044
open_rentals,1132
payments,106049
revenue,471740.51


... (2 outputs left out)

wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
15 of 15 green
wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
15 of 15 green
Drop 8 landed: 2006-10. Control totals now:


metric,value
drop,8
customers,599
films,1000
inventory,4581
rentals_landed,126830
payments_landed,120000
rentals,136044
open_rentals,1046
payments,136049
revenue,606116.51


wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
15 of 15 green
wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
15 of 15 green
15 of 15 green


,wave,job,status,seconds,run page
0,1,a1_silver_customers,GREEN,43,run page (redacted)
1,1,a2_silver_films,GREEN,42,run page (redacted)
2,1,a3_silver_inventory,GREEN,42,run page (redacted)
3,2,a4_stream_rentals,GREEN,37,run page (redacted)
4,2,a5_stream_payments,GREEN,35,run page (redacted)
5,3,a6_silver_rentals,GREEN,42,run page (redacted)
6,3,a7_silver_payments,GREEN,25,run page (redacted)
7,3,e1_enriched_customers_scd2,GREEN,34,run page (redacted)
8,3,e2_enriched_films_scd2,GREEN,35,run page (redacted)
9,3,e3_enriched_inventory_scd2,GREEN,34,run page (redacted)


In [0]:
_current_drop()

8

## Step 12 · Finish
With all eight drops landed, run the fleet twice more.

**EVIDENCE E2:** screenshot the last status table: **15 of 15 green** after drop 8, with your user icon showing.

**EVIDENCE E3:** open **Jobs & Pipelines**, job **A7 a8_gold_revenue_by_month_store**, its latest run, and the query profile of its write (**See performance** / **View query profile**). Screenshot it, and in your write-up say: how many rows were read from the payment table, which join was a broadcast, and which step took the most time.

In [0]:
run_fleet()

wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
15 of 15 green


,wave,job,status,seconds,run page
0,1,a1_silver_customers,GREEN,46,run page (redacted)
1,1,a2_silver_films,GREEN,44,run page (redacted)
2,1,a3_silver_inventory,GREEN,43,run page (redacted)
3,2,a4_stream_rentals,GREEN,40,run page (redacted)
4,2,a5_stream_payments,GREEN,37,run page (redacted)
5,3,a6_silver_rentals,GREEN,38,run page (redacted)
6,3,a7_silver_payments,GREEN,23,run page (redacted)
7,3,e1_enriched_customers_scd2,GREEN,34,run page (redacted)
8,3,e2_enriched_films_scd2,GREEN,31,run page (redacted)
9,3,e3_enriched_inventory_scd2,GREEN,30,run page (redacted)


In [0]:
run_fleet()

wave 1: a1_silver_customers, a2_silver_films, a3_silver_inventory
wave 2: a4_stream_rentals, a5_stream_payments
wave 3: a6_silver_rentals, a7_silver_payments, e1_enriched_customers_scd2, e2_enriched_films_scd2, e3_enriched_inventory_scd2
wave 4: a8_gold_revenue_by_month_store, a9_gold_late_returns, a10_gold_top_films, a12_gold_customer_journeys
wave 5: a11_quality_gate
15 of 15 green


,wave,job,status,seconds,run page
0,1,a1_silver_customers,GREEN,46,run page (redacted)
1,1,a2_silver_films,GREEN,45,run page (redacted)
2,1,a3_silver_inventory,GREEN,42,run page (redacted)
3,2,a4_stream_rentals,GREEN,37,run page (redacted)
4,2,a5_stream_payments,GREEN,36,run page (redacted)
5,3,a6_silver_rentals,GREEN,38,run page (redacted)
6,3,a7_silver_payments,GREEN,28,run page (redacted)
7,3,e1_enriched_customers_scd2,GREEN,36,run page (redacted)
8,3,e2_enriched_films_scd2,GREEN,32,run page (redacted)
9,3,e3_enriched_inventory_scd2,GREEN,32,run page (redacted)


In [0]:
#EC1
# EC1: the whole fleet as ONE job, with each task waiting only for the jobs whose tables it reads
DEPS = {
    "a1_silver_customers": [], "a2_silver_films": [], "a3_silver_inventory": [],
    "a4_stream_rentals": [], "a5_stream_payments": [],
    "a6_silver_rentals": ["a4_stream_rentals"],
    "a7_silver_payments": ["a5_stream_payments"],
    "e1_enriched_customers_scd2": [], "e2_enriched_films_scd2": [], "e3_enriched_inventory_scd2": [],
    "a8_gold_revenue_by_month_store": ["a6_silver_rentals", "a7_silver_payments", "a3_silver_inventory"],
    "a9_gold_late_returns": ["a6_silver_rentals", "a3_silver_inventory", "a2_silver_films"],
    "a10_gold_top_films": ["a6_silver_rentals", "a3_silver_inventory", "a2_silver_films"],
    "a12_gold_customer_journeys": ["e1_enriched_customers_scd2", "a6_silver_rentals", "a7_silver_payments"],
    "a11_quality_gate": ["a1_silver_customers", "e2_enriched_films_scd2", "e3_enriched_inventory_scd2",
                         "a8_gold_revenue_by_month_store", "a9_gold_late_returns",
                         "a10_gold_top_films", "a12_gold_customer_journeys"],
}
assert set(DEPS) == set(PIPELINES)

PIPE_NAME = "A7 fleet pipeline (EC1)"
for old in w.jobs.list(name=PIPE_NAME):
    w.jobs.delete(job_id=old.job_id)

tasks = [jobs.Task(task_key=n,
                   notebook_task=jobs.NotebookTask(notebook_path=f"{FOLDER}/{n}", source=jobs.Source.WORKSPACE),
                   depends_on=[jobs.TaskDependency(task_key=d) for d in DEPS[n]],
                   max_retries=0)
         for n in PIPELINES]
pipe = w.jobs.create(name=PIPE_NAME, max_concurrent_runs=1, tasks=tasks,
                     tags={"course": "adsp31011", "lab": "a7", "ec": "1"})
print("created job", pipe.job_id)

wt = w.jobs.run_now(job_id=pipe.job_id)
run_id = wt.response.run_id if hasattr(wt, "response") else wt.run_id
_wait([run_id], timeout_min=40)
run = w.jobs.get_run(run_id=run_id)
print("result:", run.state.result_state.value if run.state.result_state else run.state.life_cycle_state.value)
print("run page:", run.run_page_url)

created job 584100363507113
result: SUCCESS
run page: https://<workspace-host>/?o=<workspace-id>#job/584100363507113/run/263991603011515


In [0]:
#EC2 code to check the numbers first:
for t in ["a7_bronze.rental_raw", "a7_bronze.rental_stream", "a7_bronze.payment_stream",
          "a7_silver.customers", "a7_silver.films", "a7_silver.inventory", "a7_silver.rental", "a7_silver.payment",
          "a7_silver_enriched.customers_scd2", "a7_silver_enriched.films_scd2", "a7_silver_enriched.inventory_scd2",
          "a7_gold.revenue_by_month_store", "a7_gold.late_returns", "a7_gold.top_films", "a7_gold.customer_journeys"]:
    print(f"{t:40s} {spark.table(f'workspace.{t}').count():>9,}") 

a7_bronze.rental_raw                        16,044
a7_bronze.rental_stream                    126,830
a7_bronze.payment_stream                   120,000
a7_silver.customers                            599
a7_silver.films                              1,000
a7_silver.inventory                          4,581
a7_silver.rental                           136,044
a7_silver.payment                          136,049
a7_silver_enriched.customers_scd2              796
a7_silver_enriched.films_scd2                1,289
a7_silver_enriched.inventory_scd2            5,303
a7_gold.revenue_by_month_store                  26
a7_gold.late_returns                        64,707
a7_gold.top_films                              624
a7_gold.customer_journeys                      796
